In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from src.parser import parse_fields, extract_block_id, to_timestamp, build_miner, mine_templates
from src.features import build_sessions, attach_labels, count_vector, apply_tfidf

df = parse_fields('../data/raw/HDFS.log')
df['BlockId'] = df['content'].apply(extract_block_id)

miner = build_miner('../drain3.ini', '../models/drain_state_full.bin')
df = mine_templates(df, miner)

print("Templates:", df['EventId'].nunique())

In [ ]:
sessions = build_sessions(df)
sessions = attach_labels(sessions, '../data/raw/anomaly_label.csv')

X_counts, event_names = count_vector(sessions)
X = apply_tfidf(X_counts)
y = sessions['y'].values

np.save('../data/features/X_full.npy', X)
np.save('../data/features/y_full.npy', y)
np.save('../data/features/X_counts_full.npy', X_counts)
print("Shape:", X.shape, "| Anomalies:", y.sum())